# LLaMA-3.1-8B-Instruct FineTuning — NL to CNL Translation

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import os
import json
from pathlib import Path
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import EarlyStoppingCallback
import trl
from trl import SFTTrainer, SFTConfig
import torch
import peft
from peft import LoraConfig

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)


Dataset Generation 

In [ ]:
## wrap your dataset loading and splitting

def load_data(path, test_size=0.1, seed=42):
    def create_conversation(sample):
        return {
            "messages": [
                {"role": "system",
                 "content": "You are an expert in Translating the Natural language (NL) into Controlled Natural Language (CNL) translation. Always provide precise, syntactically correct translations of NL into CNL."},
                {"role": "user", "content": f"Translate the following natural language to controlled natural language: {sample['data_dict']['NL_V2']} "},
                {"role": "assistant", "content": sample["data_dict"]["CNL_V2"]},
            ]
        }

    # Load and transform
    dataset = load_dataset("json", data_files=path, split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
    print("Dataset converted to conversational format.")

    # Split into train and test
    split_dataset = dataset.train_test_split(test_size=test_size, seed=seed)
    train_dataset = split_dataset["train"]
    test_dataset = split_dataset["test"]

    # Optional: shuffle train (though train_test_split doesn't shuffle by default, but map preserves order)
    train_dataset = train_dataset.shuffle(seed=seed)

    return train_dataset, test_dataset



In [ ]:

dataset_file = "Path/To/Your/train_data.json" 
train_data, test_data = load_data(dataset_file, test_size=0.1)

# Optional: inspect an example
print("Example from train set:")
print(train_data[2])

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size:  {len(test_data)}")

FineTuning

In [ ]:
base_model_name = "meta-llama/Llama-3.1-8B-Instruct"

def load_prepare_model(model_name=base_model_name):
    """
    Loads and prepares the Llama-3.1-8B-Instruct model and tokenizer.

    Args:
        model_name (str): Hugging Face model identifier.

    Returns:
        model: Loaded causal language model.
        tokenizer: Corresponding tokenizer with padding configured.
    """
    # float16 for LLaMA (change if your GPU supports it)
    compute_dtype = torch.float16

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=compute_dtype,     
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # LLaMA-3 has no pad token by default — use eos token
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer


In [ ]:
base_model = base_model_name
model, tokenizer = load_prepare_model(model_name=base_model)

## Dataset Pre-processing for Completion-Only Loss

In [ ]:
# ── Dataset pre-processing for completion_only_loss ──────────────────────────
# Converts messages → prompt+completion string format.
# LLaMA-3 chat template uses <|start_header_id|>assistant<|end_header_id|>
# as the assistant turn header.

LLAMA3_ASSISTANT_HEADER = "<|start_header_id|>assistant<|end_header_id|>\n\n"

def preprocess_to_completion_format(dataset, tokenizer):
    """
    Converts the messages dataset into prompt+completion string pairs.

    Builds the FULL conversation string in ONE apply_chat_template call to avoid
    BOS/special-token boundary mismatches when tokenizing prompt vs prompt+completion
    separately. Then splits at the last assistant header occurrence.
    """
    def convert(sample):
        full_text = tokenizer.apply_chat_template(
            sample["messages"],          # system + user + assistant
            tokenize=False,
            add_generation_prompt=False,
        )

        # Split at the last assistant header
        split_idx = full_text.rfind(LLAMA3_ASSISTANT_HEADER)
        if split_idx == -1:
            raise ValueError(
                f"LLaMA-3 assistant header not found in rendered template.\n"
                f"Full text snippet: {full_text[:300]}\n"
                "Check that your tokenizer is LLaMA-3.1-8B-Instruct."
            )

        prompt     = full_text[:split_idx + len(LLAMA3_ASSISTANT_HEADER)]
        completion = full_text[split_idx + len(LLAMA3_ASSISTANT_HEADER):]

        return {"prompt": prompt, "completion": completion}

    return dataset.map(convert, remove_columns=dataset.column_names)


# Pre-process both splits
train_data_processed = preprocess_to_completion_format(train_data, tokenizer)
test_data_processed  = preprocess_to_completion_format(test_data,  tokenizer)

# Verify — inspect one sample
sample = train_data_processed[0]
print("── PROMPT (last 150 chars) ──────────────────────────────────────")
print(repr(sample["prompt"][-150:]))
print("\n── COMPLETION ───────────────────────────────────────────────────")
print(repr(sample["completion"]))


In [ ]:

max_seq_length = 1024

def create_trainer(model, tokenizer, train_dataset, eval_dataset=None,
                   max_seq_length=max_seq_length, num_epochs=2, early_stopping_patience=3):
    """
    Creates and returns an SFTTrainer instance configured for LLaMA-3.1-8B fine-tuning.

    Dataset format expected: {"prompt": "...", "completion": "..."}
    """
    peft_config = LoraConfig(
        r=8,    #16, 8
        lora_alpha=16,    #32, 64     # 2 * r
        lora_dropout=0.15,  #0.1, 0.15
        bias="none",
        use_rslora=True,        # stabilises training with rank scaling
        use_dora=False,         # mutually exclusive with use_rslora
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )

    training_arguments = SFTConfig(
        output_dir="./tuning_datasetwithpredicates_llama3_8b_r8",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        optim="adamw_torch",
        logging_steps=25,
        learning_rate=5e-5,  # 1e-4 , 5e-5
        weight_decay=0.05,   #0.02, 0.05
        fp16=True,              # LLaMA-3.1 uses float16
        bf16=False,
        max_grad_norm=0.3,
        max_steps=-1,
        warmup_ratio=0.05,  #0.03
        group_by_length=True,
        lr_scheduler_type="cosine",
        neftune_noise_alpha=10.0,  #5.0, 10.0
        load_best_model_at_end=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=max_seq_length,
        completion_only_loss=True,
    )

    callbacks = []
    if eval_dataset is not None and early_stopping_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_arguments,
        callbacks=callbacks,
    )

    return trainer


In [ ]:


trainer = create_trainer(
    model, tokenizer,
    train_dataset=train_data_processed,     # pre-processed prompt+completion
    eval_dataset=test_data_processed,       # not raw train_data/test_data
    num_epochs=15,
    early_stopping_patience=3
)
trainer.train()

print("Best checkpoint:", trainer.state.best_model_checkpoint)

# Save adapter
trainer.save_model("PATH/TO/SAVE/ADAPTER")  # Saves the adapter weights and training config
tokenizer.save_pretrained("PATH/TO/SAVE/ADAPTER")
print("Adapter saved to PATH/TO/SAVE/ADAPTER")


In [ ]:
trainer.state.best_model_checkpoint ## path to best checkpoint SAVED